In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("player_stats_engineered.csv")

print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

df.head()

Shape: (26669, 44)
Columns:
['id', 'slug', 'name', 'team', 'opponent', 'location', 'outcome', 'game_date', 'seconds_played', 'made_field_goals', 'attempted_field_goals', 'made_three_point_field_goals', 'attempted_three_point_field_goals', 'made_free_throws', 'attempted_free_throws', 'offensive_rebounds', 'defensive_rebounds', 'assists', 'steals', 'blocks', 'turnovers', 'personal_fouls', 'plus_minus', 'game_score', 'points', 'total_rebounds', 'minutes', 'rolling_pts_5', 'rolling_reb_5', 'rolling_ast_5', 'rolling_min_5', 'rolling_fg_pct_5', 'rolling_3p_pct_5', 'rolling_pts_10', 'rolling_reb_10', 'rolling_ast_10', 'rolling_min_10', 'rolling_fg_pct_10', 'rolling_3p_pct_10', 'home_away_pts_avg', 'home_away_reb_avg', 'home_away_ast_avg', 'rest_days', 'is_back_to_back']


,id,slug,name,team,opponent,location,outcome,game_date,seconds_played,made_field_goals,...,rolling_reb_10,rolling_ast_10,rolling_min_10,rolling_fg_pct_10,rolling_3p_pct_10,home_away_pts_avg,home_away_reb_avg,home_away_ast_avg,rest_days,is_back_to_back
0,4079,achiupr01,Precious Achiuwa,SACRAMENTO_KINGS,MINNESOTA_TIMBERWOLVES,AWAY,LOSS,2025-11-14,1179,2,...,5.000000,1.000000,18.660000,0.600000,0.000000,NaN,NaN,NaN,2.0,0
1,4422,achiupr01,Precious Achiuwa,SACRAMENTO_KINGS,SAN_ANTONIO_SPURS,AWAY,LOSS,2025-11-16,275,0,...,4.666667,0.833333,18.825000,0.586207,0.000000,4.000000,3.000000,0.00,2.0,0
2,4784,achiupr01,Precious Achiuwa,SACRAMENTO_KINGS,OKLAHOMA_CITY_THUNDER,AWAY,LOSS,2025-11-19,1572,6,...,4.142857,0.714286,16.790476,0.548387,0.000000,2.500000,2.000000,0.00,3.0,0
3,4987,achiupr01,Precious Achiuwa,SACRAMENTO_KINGS,MEMPHIS_GRIZZLIES,AWAY,LOSS,2025-11-20,1446,1,...,4.750000,1.000000,17.966667,0.534884,0.111111,6.666667,4.333333,1.00,1.0,1
4,5273,achiupr01,Precious Achiuwa,SACRAMENTO_KINGS,DENVER_NUGGETS,AWAY,WIN,2025-11-22,1417,3,...,4.777778,1.111111,18.648148,0.521739,0.090909,5.500000,4.500000,1.25,2.0,0


In [ ]:
# Convert game_date to datetime
df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce")

# Clean location just in case it has values like "Location.HOME"
df["location_clean"] = (
    df["location"]
    .astype(str)
    .str.replace("Location.", "", regex=False)
    .str.upper()
)

# Turn home/away into a numeric feature
df["is_home"] = (df["location_clean"] == "HOME").astype(int)

# Features known before the game
feature_cols = [
    "rolling_pts_5",
    "rolling_reb_5",
    "rolling_ast_5",
    "rolling_min_5",
    "rolling_fg_pct_5",
    "rolling_3p_pct_5",
    "rolling_pts_10",
    "rolling_reb_10",
    "rolling_ast_10",
    "rolling_min_10",
    "rolling_fg_pct_10",
    "rolling_3p_pct_10",
    "home_away_pts_avg",
    "home_away_reb_avg",
    "home_away_ast_avg",
    "rest_days",
    "is_back_to_back",
    "is_home",
]

# Targets we want to predict
targets = {
    "points": "points",
    "rebounds": "total_rebounds",
    "assists": "assists",
}

# Check missing columns
needed_cols = ["game_date", "slug", "name", "team", "opponent", "location"] + feature_cols + list(targets.values())
missing_cols = [col for col in needed_cols if col not in df.columns]

print("Missing columns:", missing_cols)

if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

# Sort by date for time-based split
df = df.dropna(subset=["game_date"]).sort_values("game_date").reset_index(drop=True)

print("Date range:", df["game_date"].min(), "to", df["game_date"].max())
print("Rows:", len(df))

Missing columns: []
Date range: 2025-10-21 00:00:00 to 2026-04-12 00:00:00
Rows: 26669


In [ ]:
unique_dates = sorted(df["game_date"].dropna().unique())

cutoff_index = int(len(unique_dates) * 0.75)
cutoff_date = unique_dates[cutoff_index]

train_df = df[df["game_date"] < cutoff_date].copy()
test_df = df[df["game_date"] >= cutoff_date].copy()

print("Cutoff date:", pd.to_datetime(cutoff_date).date())
print("Train rows:", train_df.shape)
print("Test rows:", test_df.shape)

Cutoff date: 2026-03-01
Train rows: (19438, 46)
Test rows: (7231, 46)


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor

import joblib
from pathlib import Path

In [ ]:
def build_linear_model():
    """
    Linear Regression baseline.
    Simple model to compare against XGBoost.
    """
    model = Pipeline([
        ("preprocess", ColumnTransformer([
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), feature_cols)
        ])),
        ("model", LinearRegression())
    ])

    return model


def build_xgb_model():
    """
    XGBoost Regressor.
    Stronger nonlinear model.
    """
    model = Pipeline([
        ("preprocess", ColumnTransformer([
            ("num", SimpleImputer(strategy="median"), feature_cols)
        ])),
        ("model", XGBRegressor(
            n_estimators=150,
            learning_rate=0.05,
            max_depth=3,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=42
        ))
    ])

    return model


def evaluate_model(y_true, y_pred):
    """
    Evaluate regression model.
    MAE = average absolute error.
    RMSE = larger errors are punished more.
    R2 = how much variance the model explains.
    """
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    return mae, rmse, r2

In [ ]:
models = {
    "linear_regression": build_linear_model,
    "xgboost": build_xgb_model,
}

# Naive baseline columns
naive_baselines = {
    "points": "rolling_pts_5",
    "rebounds": "rolling_reb_5",
    "assists": "rolling_ast_5",
}

metrics_rows = []
prediction_frames = []
trained_models = {}

for target_name, target_col in targets.items():
    print("\n==============================")
    print(f"Training target: {target_name}")
    print("==============================")

    # Drop rows where target is missing
    train_target_df = train_df.dropna(subset=[target_col]).copy()
    test_target_df = test_df.dropna(subset=[target_col]).copy()

    X_train = train_target_df[feature_cols]
    y_train = train_target_df[target_col]

    X_test = test_target_df[feature_cols]
    y_test = test_target_df[target_col]

    # ----------------------------
    # Naive baseline
    # ----------------------------
    naive_col = naive_baselines[target_name]
    naive_preds = test_target_df[naive_col].copy()

    # Fill missing baseline values with training target median
    naive_preds = naive_preds.fillna(y_train.median())

    mae, rmse, r2 = evaluate_model(y_test, naive_preds)

    metrics_rows.append({
        "target": target_name,
        "model": "naive_rolling_5",
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "train_rows": len(train_target_df),
        "test_rows": len(test_target_df),
    })

    naive_pred_df = test_target_df[
        ["game_date", "slug", "name", "team", "opponent", "location"]
    ].copy()

    naive_pred_df["target"] = target_name
    naive_pred_df["model"] = "naive_rolling_5"
    naive_pred_df["actual"] = y_test.values
    naive_pred_df["prediction"] = naive_preds.values
    naive_pred_df["error"] = naive_pred_df["prediction"] - naive_pred_df["actual"]

    prediction_frames.append(naive_pred_df)

    print(f"Naive rolling 5 MAE: {mae:.3f}")

    # ----------------------------
    # Linear Regression and XGBoost
    # ----------------------------
    for model_name, model_builder in models.items():
        print(f"Training model: {model_name}")

        model = model_builder()
        model.fit(X_train, y_train)

        preds = model.predict(X_test)

        mae, rmse, r2 = evaluate_model(y_test, preds)

        metrics_rows.append({
            "target": target_name,
            "model": model_name,
            "mae": mae,
            "rmse": rmse,
            "r2": r2,
            "train_rows": len(train_target_df),
            "test_rows": len(test_target_df),
        })

        pred_df = test_target_df[
            ["game_date", "slug", "name", "team", "opponent", "location"]
        ].copy()

        pred_df["target"] = target_name
        pred_df["model"] = model_name
        pred_df["actual"] = y_test.values
        pred_df["prediction"] = preds
        pred_df["error"] = pred_df["prediction"] - pred_df["actual"]

        prediction_frames.append(pred_df)

        trained_models[(target_name, model_name)] = model

        print(f"{model_name} MAE: {mae:.3f}")


Training target: points
Naive rolling 5 MAE: 4.933
Training model: linear_regression
linear_regression MAE: 4.759
Training model: xgboost
xgboost MAE: 4.771

Training target: rebounds
Naive rolling 5 MAE: 1.985
Training model: linear_regression
linear_regression MAE: 1.912
Training model: xgboost
xgboost MAE: 1.917

Training target: assists
Naive rolling 5 MAE: 1.442
Training model: linear_regression
linear_regression MAE: 1.392
Training model: xgboost
xgboost MAE: 1.393


In [ ]:
metrics_df = pd.DataFrame(metrics_rows)
predictions_df = pd.concat(prediction_frames, ignore_index=True)

metrics_df.sort_values(["target", "mae"])

,target,model,mae,rmse,r2,train_rows,test_rows
7,assists,linear_regression,1.392190,1.905489,0.463792,19438,7231
8,assists,xgboost,1.392782,1.906649,0.463139,19438,7231
6,assists,naive_rolling_5,1.442138,1.981270,0.420294,19438,7231
1,points,linear_regression,4.759066,6.217103,0.461863,19438,7231
2,points,xgboost,4.770961,6.230686,0.459509,19438,7231
0,points,naive_rolling_5,4.933140,6.489560,0.413663,19438,7231
4,rebounds,linear_regression,1.912104,2.529416,0.412705,19438,7231
5,rebounds,xgboost,1.917337,2.530771,0.412075,19438,7231
3,rebounds,naive_rolling_5,1.985465,2.661475,0.349780,19438,7231


In [ ]:
team_df = pd.read_csv("team_game_stats_engineered.csv")

print("Shape:", team_df.shape)
print(team_df.columns.tolist())
team_df.head()

Shape: (2644, 17)
['game_date', 'team', 'opponent', 'points_scored', 'points_allowed', 'location', 'won', 'rolling_pts_scored_5', 'rolling_pts_allowed_5', 'rolling_win_pct_5', 'rolling_pts_scored_10', 'rolling_pts_allowed_10', 'rolling_win_pct_10', 'home_away_pts_scored_avg', 'home_away_pts_allowed_avg', 'rest_days', 'is_back_to_back']


,game_date,team,opponent,points_scored,points_allowed,location,won,rolling_pts_scored_5,rolling_pts_allowed_5,rolling_win_pct_5,rolling_pts_scored_10,rolling_pts_allowed_10,rolling_win_pct_10,home_away_pts_scored_avg,home_away_pts_allowed_avg,rest_days,is_back_to_back
0,2025-10-22,ATLANTA_HAWKS,TORONTO_RAPTORS,118,138,HOME,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,2025-10-24,ATLANTA_HAWKS,ORLANDO_MAGIC,111,107,AWAY,1,118.000000,138.000000,0.000000,118.000000,138.000000,0.000000,NaN,NaN,2.0,0
2,2025-10-25,ATLANTA_HAWKS,OKLAHOMA_CITY_THUNDER,100,117,HOME,0,114.500000,122.500000,0.500000,114.500000,122.500000,0.500000,118.0,138.0,1.0,1
3,2025-10-27,ATLANTA_HAWKS,CHICAGO_BULLS,123,128,AWAY,0,109.666667,120.666667,0.333333,109.666667,120.666667,0.333333,111.0,107.0,2.0,0
4,2025-10-29,ATLANTA_HAWKS,BROOKLYN_NETS,117,112,AWAY,1,113.000000,122.500000,0.250000,113.000000,122.500000,0.250000,117.0,117.5,2.0,0


In [ ]:
team_df["game_date"] = pd.to_datetime(team_df["game_date"], errors="coerce")

team_df["location_clean"] = (
    team_df["location"]
    .astype(str)
    .str.replace("Location.", "", regex=False)
    .str.upper()
)

team_df = (
    team_df
    .dropna(subset=["game_date"])
    .sort_values(["team", "game_date"])
    .reset_index(drop=True)
)

home_rows = team_df[team_df["location_clean"] == "HOME"].copy()
away_rows = team_df[team_df["location_clean"] == "AWAY"].copy()

home_rows = home_rows.rename(columns={
    col: f"home_{col}" for col in home_rows.columns if col != "game_date"
})

away_rows = away_rows.rename(columns={
    col: f"away_{col}" for col in away_rows.columns if col != "game_date"
})

matchup_df = home_rows.merge(
    away_rows,
    left_on=["game_date", "home_team", "home_opponent"],
    right_on=["game_date", "away_opponent", "away_team"],
    how="inner"
)

matchup_df["home_team_win"] = matchup_df["home_won"]

print("Matchup shape:", matchup_df.shape)
matchup_df[["game_date", "home_team", "away_team", "home_team_win"]].head()

Matchup shape: (1322, 36)


,game_date,home_team,away_team,home_team_win
0,2025-10-22,ATLANTA_HAWKS,TORONTO_RAPTORS,0
1,2025-10-25,ATLANTA_HAWKS,OKLAHOMA_CITY_THUNDER,0
2,2025-11-04,ATLANTA_HAWKS,ORLANDO_MAGIC,1
3,2025-11-07,ATLANTA_HAWKS,TORONTO_RAPTORS,0
4,2025-11-08,ATLANTA_HAWKS,LOS_ANGELES_LAKERS,1


In [ ]:
base_team_features = [
    "rolling_pts_scored_5",
    "rolling_pts_allowed_5",
    "rolling_win_pct_5",
    "rolling_pts_scored_10",
    "rolling_pts_allowed_10",
    "rolling_win_pct_10",
    "home_away_pts_scored_avg",
    "home_away_pts_allowed_avg",
    "rest_days",
    "is_back_to_back",
]

win_feature_cols = []

for feature in base_team_features:
    home_col = f"home_{feature}"
    away_col = f"away_{feature}"

    if home_col in matchup_df.columns and away_col in matchup_df.columns:
        diff_col = f"diff_{feature}"
        matchup_df[diff_col] = matchup_df[home_col] - matchup_df[away_col]
        win_feature_cols.extend([home_col, away_col, diff_col])

print("Number of features:", len(win_feature_cols))
print(win_feature_cols)

Number of features: 30
['home_rolling_pts_scored_5', 'away_rolling_pts_scored_5', 'diff_rolling_pts_scored_5', 'home_rolling_pts_allowed_5', 'away_rolling_pts_allowed_5', 'diff_rolling_pts_allowed_5', 'home_rolling_win_pct_5', 'away_rolling_win_pct_5', 'diff_rolling_win_pct_5', 'home_rolling_pts_scored_10', 'away_rolling_pts_scored_10', 'diff_rolling_pts_scored_10', 'home_rolling_pts_allowed_10', 'away_rolling_pts_allowed_10', 'diff_rolling_pts_allowed_10', 'home_rolling_win_pct_10', 'away_rolling_win_pct_10', 'diff_rolling_win_pct_10', 'home_home_away_pts_scored_avg', 'away_home_away_pts_scored_avg', 'diff_home_away_pts_scored_avg', 'home_home_away_pts_allowed_avg', 'away_home_away_pts_allowed_avg', 'diff_home_away_pts_allowed_avg', 'home_rest_days', 'away_rest_days', 'diff_rest_days', 'home_is_back_to_back', 'away_is_back_to_back', 'diff_is_back_to_back']


In [ ]:
matchup_df = matchup_df.sort_values("game_date").reset_index(drop=True)

unique_dates = sorted(matchup_df["game_date"].dropna().unique())

cutoff_index = int(len(unique_dates) * 0.75)
cutoff_date = unique_dates[cutoff_index]

train_df = matchup_df[matchup_df["game_date"] < cutoff_date].copy()
test_df = matchup_df[matchup_df["game_date"] >= cutoff_date].copy()

print("Cutoff date:", pd.to_datetime(cutoff_date).date())
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Cutoff date: 2026-04-06
Train shape: (1173, 46)
Test shape: (149, 46)


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss

from xgboost import XGBClassifier


def build_logistic_model():
    return Pipeline([
        ("preprocess", ColumnTransformer([
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), win_feature_cols)
        ])),
        ("model", LogisticRegression(max_iter=1000, random_state=42))
    ])


def build_xgb_classifier():
    return Pipeline([
        ("preprocess", ColumnTransformer([
            ("num", SimpleImputer(strategy="median"), win_feature_cols)
        ])),
        ("model", XGBClassifier(
            n_estimators=150,
            learning_rate=0.05,
            max_depth=3,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=42
        ))
    ])


def evaluate_win_model(y_true, probabilities):
    predicted = (probabilities >= 0.5).astype(int)

    accuracy = accuracy_score(y_true, predicted)
    loss = log_loss(y_true, probabilities)

    return accuracy, loss

In [ ]:
X_train = train_df[win_feature_cols]
y_train = train_df["home_team_win"]

X_test = test_df[win_feature_cols]
y_test = test_df["home_team_win"]

win_models = {
    "logistic_regression": build_logistic_model(),
    "xgboost_classifier": build_xgb_classifier(),
}

win_metrics = []
win_predictions = []
trained_win_models = {}

# Naive baseline: 50/50 probability
naive_probs = np.full(len(y_test), 0.50)
naive_accuracy, naive_log_loss = evaluate_win_model(y_test, naive_probs)

win_metrics.append({
    "model": "naive_50_50",
    "accuracy": naive_accuracy,
    "log_loss": naive_log_loss,
    "train_rows": len(train_df),
    "test_rows": len(test_df)
})

print("Naive 50/50")
print("Accuracy:", naive_accuracy)
print("Log loss:", naive_log_loss)

for model_name, model in win_models.items():
    print("\nTraining:", model_name)

    model.fit(X_train, y_train)

    probs = model.predict_proba(X_test)[:, 1]

    accuracy, loss = evaluate_win_model(y_test, probs)

    win_metrics.append({
        "model": model_name,
        "accuracy": accuracy,
        "log_loss": loss,
        "train_rows": len(train_df),
        "test_rows": len(test_df)
    })

    pred_df = test_df[["game_date", "home_team", "away_team", "home_team_win"]].copy()
    pred_df["model"] = model_name
    pred_df["home_win_probability"] = probs
    pred_df["away_win_probability"] = 1 - probs
    pred_df["predicted_home_win"] = (probs >= 0.5).astype(int)
    pred_df["predicted_winner"] = np.where(
        pred_df["predicted_home_win"] == 1,
        pred_df["home_team"],
        pred_df["away_team"]
    )

    win_predictions.append(pred_df)

    trained_win_models[model_name] = model

    print("Accuracy:", accuracy)
    print("Log loss:", loss)

win_metrics_df = pd.DataFrame(win_metrics)
win_predictions_df = pd.concat(win_predictions, ignore_index=True)

win_metrics_df.sort_values("log_loss")

Naive 50/50
Accuracy: 0.6040268456375839
Log loss: 0.6931471805599454

Training: logistic_regression
Accuracy: 0.6577181208053692
Log loss: 0.6076370341427713

Training: xgboost_classifier
Accuracy: 0.6577181208053692
Log loss: 0.6333610326353786


,model,accuracy,log_loss,train_rows,test_rows
1,logistic_regression,0.657718,0.607637,1173,149
2,xgboost_classifier,0.657718,0.633361,1173,149
0,naive_50_50,0.604027,0.693147,1173,149


In [ ]:
def predict_existing_matchup_win_probability(game_date, home_team, away_team, model_name="xgboost_classifier"):
    game_date = pd.to_datetime(game_date)

    game_row = matchup_df[
        (matchup_df["game_date"] == game_date)
        & (matchup_df["home_team"] == home_team)
        & (matchup_df["away_team"] == away_team)
    ].copy()

    if game_row.empty:
        print("No matchup found.")
        print("Check the date/team spelling.")
        return None

    model = trained_win_models[model_name]

    home_prob = model.predict_proba(game_row[win_feature_cols])[:, 1][0]
    away_prob = 1 - home_prob

    predicted_winner = home_team if home_prob >= 0.5 else away_team
    actual_winner = home_team if game_row["home_team_win"].iloc[0] == 1 else away_team

    result = {
        "game_date": game_date.date(),
        "home_team": home_team,
        "away_team": away_team,
        "home_win_probability": round(float(home_prob), 4),
        "away_win_probability": round(float(away_prob), 4),
        "predicted_winner": predicted_winner,
        "actual_winner": actual_winner
    }

    return result

In [ ]:
sample_matchups = matchup_df[["game_date", "home_team", "away_team", "home_team_win"]].copy()
sample_matchups.tail(20)

,game_date,home_team,away_team,home_team_win
1302,2026-05-13,DETROIT_PISTONS,CLEVELAND_CAVALIERS,0
1303,2026-05-15,MINNESOTA_TIMBERWOLVES,SAN_ANTONIO_SPURS,0
1304,2026-05-15,CLEVELAND_CAVALIERS,DETROIT_PISTONS,0
1305,2026-05-17,DETROIT_PISTONS,CLEVELAND_CAVALIERS,0
1306,2026-05-18,OKLAHOMA_CITY_THUNDER,SAN_ANTONIO_SPURS,0
1307,2026-05-19,NEW_YORK_KNICKS,CLEVELAND_CAVALIERS,1
1308,2026-05-20,OKLAHOMA_CITY_THUNDER,SAN_ANTONIO_SPURS,1
1309,2026-05-21,NEW_YORK_KNICKS,CLEVELAND_CAVALIERS,1
1310,2026-05-22,SAN_ANTONIO_SPURS,OKLAHOMA_CITY_THUNDER,0
1311,2026-05-23,CLEVELAND_CAVALIERS,NEW_YORK_KNICKS,0


In [ ]:
def predict_player_stats_for_matchup(game_date, home_team, away_team, model_name="xgboost", min_minutes=10):
    """
    Predict player points, rebounds, and assists for an existing matchup.

    This uses the already-trained player models:
    trained_models[("points", model_name)]
    trained_models[("rebounds", model_name)]
    trained_models[("assists", model_name)]

    Important:
    This only works for games that already exist in player_stats_engineered.csv,
    because the engineered rolling features already exist for those rows.
    """

    game_date = pd.to_datetime(game_date)

    matchup_players = df[
        (df["game_date"] == game_date)
        &
        (
            ((df["team"] == home_team) & (df["opponent"] == away_team))
            |
            ((df["team"] == away_team) & (df["opponent"] == home_team))
        )
    ].copy()

    if matchup_players.empty:
        print("No player rows found for this matchup.")
        print("Check the date and team names.")
        return None

    result = matchup_players[
        [
            "game_date",
            "slug",
            "name",
            "team",
            "opponent",
            "location",
            "rolling_min_5",
            "points",
            "total_rebounds",
            "assists",
        ]
    ].copy()

    # Predict each player stat target
    points_model = trained_models[("points", model_name)]
    rebounds_model = trained_models[("rebounds", model_name)]
    assists_model = trained_models[("assists", model_name)]

    result["predicted_points"] = points_model.predict(matchup_players[feature_cols])
    result["predicted_rebounds"] = rebounds_model.predict(matchup_players[feature_cols])
    result["predicted_assists"] = assists_model.predict(matchup_players[feature_cols])

    # Rename actual stat columns
    result = result.rename(columns={
        "points": "actual_points",
        "total_rebounds": "actual_rebounds",
        "assists": "actual_assists",
    })

    # Keep likely rotation players only
    result = result[result["rolling_min_5"].fillna(0) >= min_minutes]

    # Make values cleaner
    result["predicted_points"] = result["predicted_points"].round(1)
    result["predicted_rebounds"] = result["predicted_rebounds"].round(1)
    result["predicted_assists"] = result["predicted_assists"].round(1)

    result = result.sort_values("predicted_points", ascending=False).reset_index(drop=True)

    return result

In [ ]:
def predict_full_matchup_report(game_date, home_team, away_team):
    """
    Full matchup report:
    - win probability
    - predicted winner
    - player stat-line projections
    """

    print("===================================")
    print("NBA Predictive Modeling Report")
    print("===================================")
    print(f"Matchup: {away_team} at {home_team}")
    print(f"Date: {game_date}")
    print()

    # Win probability
    win_result = predict_existing_matchup_win_probability(
        game_date=game_date,
        home_team=home_team,
        away_team=away_team,
        model_name="xgboost_classifier"
    )

    if win_result is not None:
        print("Win Probability")
        print("----------------")
        print(f"{home_team}: {win_result['home_win_probability'] * 100:.1f}%")
        print(f"{away_team}: {win_result['away_win_probability'] * 100:.1f}%")
        print(f"Predicted winner: {win_result['predicted_winner']}")
        print(f"Actual winner: {win_result['actual_winner']}")
        print()

    # Player stat lines
    player_result = predict_player_stats_for_matchup(
        game_date=game_date,
        home_team=home_team,
        away_team=away_team,
        model_name="xgboost",
        min_minutes=10
    )

    if player_result is not None:
        print("Projected Player Stat Lines")
        print("----------------------------")

        display_cols = [
            "name",
            "team",
            "predicted_points",
            "predicted_rebounds",
            "predicted_assists",
            "actual_points",
            "actual_rebounds",
            "actual_assists",
        ]

        display(player_result[display_cols])

    return win_result, player_result

In [ ]:
#2026-03-01 and after for test split
win_result, player_result = predict_full_matchup_report(
    game_date="2026-04-07",
    home_team="BOSTON_CELTICS",
    away_team="CHARLOTTE_HORNETS"
)

NBA Predictive Modeling Report
Matchup: CHARLOTTE_HORNETS at BOSTON_CELTICS
Date: 2026-04-07

Win Probability
----------------
BOSTON_CELTICS: 60.8%
CHARLOTTE_HORNETS: 39.2%
Predicted winner: BOSTON_CELTICS
Actual winner: BOSTON_CELTICS

Projected Player Stat Lines
----------------------------


,name,team,predicted_points,predicted_rebounds,predicted_assists,actual_points,actual_rebounds,actual_assists
0,Jaylen Brown,BOSTON_CELTICS,29.000000,6.0,5.6,35,9,3
1,Jayson Tatum,BOSTON_CELTICS,22.299999,10.4,6.0,23,5,4
2,LaMelo Ball,CHARLOTTE_HORNETS,19.700001,4.7,6.6,36,5,6
3,Brandon Miller,CHARLOTTE_HORNETS,18.700001,4.8,3.4,20,0,2
4,Kon Knueppel,CHARLOTTE_HORNETS,17.500000,5.2,3.4,13,5,5
5,Payton Pritchard,BOSTON_CELTICS,16.500000,3.4,4.1,12,4,3
6,Miles Bridges,CHARLOTTE_HORNETS,15.500000,5.9,2.9,13,12,4
7,Derrick White,BOSTON_CELTICS,14.200000,4.5,4.5,12,2,3
8,Neemias Queta,BOSTON_CELTICS,12.000000,9.2,2.5,12,5,3
9,Nikola Vučević,BOSTON_CELTICS,9.800000,6.2,2.1,2,7,2
